In [3]:
import open3d as o3d
import numpy as np
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(np.random.rand(100, 3))
o3d.visualization.draw_geometries([pcd])

To get the dimensions and scale of a point cloud model in Python, you can use libraries like numpy for calculations and open3d or pyntcloud for handling point clouds. Here's how you can do it:

In [3]:
import open3d as o3d
import numpy as np

# Load your point cloud file (supports .ply, .pcd, .xyz, etc.)
pcd = o3d.io.read_point_cloud("./ply_files/test point cloud shell_1.ply")  # Replace with your file path

# Get points as a numpy array
points = np.asarray(pcd.points)

# Calculate dimensions (bounding box)
min_coords = np.min(points, axis=0)
max_coords = np.max(points, axis=0)
dimensions = max_coords - min_coords

print(f"Model dimensions (width, height, depth): {dimensions}")
print(f"Scale (max dimension): {np.max(dimensions)}")

# Optional: Visualize
o3d.visualization.draw_geometries([pcd])

Model dimensions (width, height, depth): [180.00001526 120.00001526 100.00000763]
Scale (max dimension): 180.00001525878906


## Main program

In [1]:
import numpy as np
import open3d as o3d  # 3D point cloud processing
import math  # Mathematical functions
import random  # Random number generation
import time  # Time-related functions

class PointCloud:
    def __init__(self, file_path):
        # Load point cloud from file
        self.pcd = o3d.io.read_point_cloud(file_path)
        # Check if point cloud is valid
        if not self.pcd.has_points():
            raise ValueError("Point cloud is empty")
        # Convert points to NumPy array
        self.points = np.asarray(self.pcd.points)
        # Build KDTree for efficient neighbor searches
        self.kdtree = o3d.geometry.KDTreeFlann(self.pcd)
        # Calculate centroid (mean position) of points
        self.centroid = np.mean(self.points, axis=0)
        # Get axis-aligned bounding box
        self.bbox = self.pcd.get_axis_aligned_bounding_box()
        # Calculate diagonal length of bounding box
        self.bbox_diag = np.linalg.norm(np.asarray(self.bbox.get_max_bound()) - np.asarray(self.bbox.get_min_bound()))

    def get_points(self):
        return self.points

    def get_centroid(self):
        return self.centroid

    def get_bbox_diagonal(self):
        return self.bbox_diag

class Ball:
    def __init__(self, position, direction, radius, collision_margin):
        # Initialize ball position as NumPy array
        self.position = np.array(position, dtype=np.float64)
        # Initialize and normalize direction vector
        self.direction = np.array(direction, dtype=np.float64)
        self.direction /= np.linalg.norm(self.direction)
        # Physical properties
        self.radius = radius
        self.collision_margin = collision_margin  # Extra margin around ball

    def set_position(self, position):
        self.position = np.array(position, dtype=np.float64)

    def set_direction(self, direction):
        # Update direction vector and normalize
        self.direction = np.array(direction, dtype=np.float64)
        self.direction /= np.linalg.norm(self.direction)

class ScatterSimulator:
    def __init__(self, point_cloud, ball_radius, collision_margin_ratio=0.5, outer_radius_multiplier=10.0):
        # Initialize simulation parameters
        self.point_cloud = point_cloud
        self.ball_radius = ball_radius
        self.collision_margin = ball_radius * collision_margin_ratio  # Safety margin around ball
        self.outer_radius = outer_radius_multiplier * point_cloud.get_bbox_diagonal()  # Escape boundary radius
        self.origin = None  # Starting point for balls
        self.inner_surface_points = []  # Collected inner surface points
        self.escape_count = 0  # Count of escaped balls
        self.ball_count = 0  # Total balls simulated

    def find_origin(self):
        # Find a starting point inside the surface
        centroid = self.point_cloud.get_centroid()
        k = 50  # Number of neighbors to consider
        _, indices, _ = self.point_cloud.kdtree.search_knn_vector_3d(centroid, k)
        neighbor_points = self.point_cloud.points[indices]
        # Calculate vector from centroid to neighbors
        vectors = neighbor_points - centroid
        avg_vector = np.mean(vectors, axis=0)
        # Set origin slightly inside the point cloud
        self.origin = centroid - 0.1 * avg_vector
        return self.origin

    def compute_normal(self, point, radius):
        # Compute surface normal at a point using PCA
        [k, idx, _] = self.point_cloud.kdtree.search_radius_vector_3d(point, radius)
        if k < 3:  # Insufficient points for normal estimation
            return None
        # Get points within neighborhood
        points_norm = self.point_cloud.points[idx]
        # Compute covariance matrix
        cov = np.cov(points_norm.T)
        # Eigen decomposition (PCA)
        eigenvalues, eigenvectors = np.linalg.eigh(cov)
        # Return smallest eigenvector (normal direction)
        return eigenvectors[:, 0]

    def move_until_collision_or_escape(self, ball):
        # Set movement parameters
        t_max = 10.0 * ball.radius  # Maximum movement distance
        r = ball.radius + ball.collision_margin  # Effective collision radius
        current_pos = ball.position
        direction = ball.direction

        # Search for candidate points within possible collision range
        [k, idx, _] = self.point_cloud.kdtree.search_radius_vector_3d(current_pos, r + t_max)
        if k == 0:  # No candidate points
            # Move to maximum possible position
            new_position = current_pos + t_max * direction
            # Check if ball escaped defined boundary
            if np.linalg.norm(new_position - self.origin) > self.outer_radius:
                return 'escape', None, None
            else:
                ball.set_position(new_position)
                return 'moved', None, None

        # Examine candidate points for collisions
        candidate_points = self.point_cloud.points[idx]
        t_min = float('inf')  # Initialize with large value
        collision_point = None

        # Quadratic equation solution for ray-sphere intersection
        for p in candidate_points:
            A = current_pos  # Ray origin
            d = direction   # Ray direction
            Ap = A - p      # Vector from point to ray origin
            a = 1.0        # Coefficient a = ||d||² (unit vector)
            b = 2.0 * np.dot(d, Ap)
            c = np.dot(Ap, Ap) - r * r
            discriminant = b * b - 4 * a * c
            # Check real solutions
            if discriminant < 0:
                continue
            # Calculate two possible solutions
            sqrt_disc = math.sqrt(discriminant)
            t1 = (-b - sqrt_disc) / (2 * a)
            t2 = (-b + sqrt_disc) / (2 * a)
            for t_candidate in [t1, t2]:
                # Find smallest positive t value
                if 0 < t_candidate < t_min:
                    t_min = t_candidate
                    collision_point = p

        if t_min < t_max:  # Collision occurred
            new_position = current_pos + t_min * direction
            ball.set_position(new_position)
            return 'collision', collision_point, new_position
        else:  # No collision in range
            new_position = current_pos + t_max * direction
            ball.set_position(new_position)
            # Check escape condition
            if np.linalg.norm(new_position - self.origin) > self.outer_radius:
                return 'escape', None, None
            else:
                return 'moved', None, None

    def simulate_ball(self):
        self.ball_count += 1
        # Initialize random direction
        initial_direction = np.random.randn(3)
        initial_direction /= np.linalg.norm(initial_direction)
        # Create new ball instance
        ball = Ball(self.origin, initial_direction, self.ball_radius, self.collision_margin)
        collision_count = 0

        # Main ball simulation loop
        while True:
            result, collision_point, new_position = self.move_until_collision_or_escape(ball)
            if result == 'escape':  # Ball escaped
                self.escape_count += 1
                break
            if result == 'collision':  # Collision occurred
                collision_count += 1
                # Record collision point as inner surface candidate
                self.inner_surface_points.append(collision_point)
                # Compute surface normal at collision point
                normal = self.compute_normal(collision_point, 3 * ball.radius)
                # Fallback if normal computation fails
                if normal is None:
                    normal = np.random.randn(3)
                    normal /= np.linalg.norm(normal)
                # Orient normal towards the ball
                to_ball = ball.position - collision_point
                if np.dot(normal, to_ball) < 0:
                    normal = -normal
                # Calculate reflection direction
                v_old = ball.direction
                v_new = v_old - 2 * np.dot(v_old, normal) * normal
                # Add random perturbation to direction
                perturbation = 0.1 * np.random.randn(3)
                v_new += perturbation
                v_new /= np.linalg.norm(v_new)
                # Update ball state
                ball.set_direction(v_new)
                # Move ball away from collision point
                ball.set_position(collision_point + (ball.radius + ball.collision_margin) * v_new)

    def run(self, num_balls=10):
        # Ensure origin is initialized
        self.find_origin()
        # Simulate multiple balls
        for _ in range(num_balls):
            self.simulate_ball()
        return np.array(self.inner_surface_points)  # Return collected inner points

def visualize_process(point_cloud, origin, inner_points, update_interval=5):
    # Initialize visualizer
    vis = o3d.visualization.Visualizer()
    vis.create_window()
    
    # Configure original point cloud
    original_pcd = point_cloud.pcd
    original_pcd.paint_uniform_color([0.9, 0.9, 0.9])  # Dark gray
    vis.add_geometry(original_pcd)
    
    # Create origin indicator
    origin_sphere = o3d.geometry.TriangleMesh.create_sphere(radius=0.02)
    origin_sphere.paint_uniform_color([1, 0, 0])  # Red
    origin_sphere.translate(origin)
    vis.add_geometry(origin_sphere)
    
    # Initialize container for inner surface points
    inner_pcd = o3d.geometry.PointCloud()
    inner_pcd.points = o3d.utility.Vector3dVector([origin])  # Start empty
    inner_pcd.paint_uniform_color([0, 1, 0])  # Green
    vis.add_geometry(inner_pcd)
    
    # Track last update time
    last_update = time.time()
    
    return vis, inner_pcd, last_update  # Return visualization objects

def main(ply_file, num_balls=100, ball_radius_factor=0.01):
    # Load point cloud
    point_cloud = PointCloud(ply_file)
    # Calculate ball radius based on point cloud size
    bbox_diag = point_cloud.get_bbox_diagonal()
    ball_radius = ball_radius_factor * bbox_diag

    # Initialize simulator
    simulator = ScatterSimulator(point_cloud, ball_radius)
    # Ensure origin is calculated
    simulator.find_origin()
    simulator.inner_surface_points.append(simulator.origin)

    # Setup visualization
    vis, inner_pcd, last_update = visualize_process(point_cloud, simulator.origin, simulator.inner_surface_points)
 
    # Main simulation loop
    for i in range(num_balls):
        # simulator.simulate_ball()
        current_time = time.time()
        # Update visualization periodically
        '''if current_time - last_update >= 5 or i == 0:
            inner_pcd.points = o3d.utility.Vector3dVector(np.array(simulator.inner_surface_points))
            vis.update_geometry(inner_pcd)
            vis.poll_events()
            vis.update_renderer()
            last_update = current_time'''
        print(i)
        if i > 1000:
            print(f"i > 1000, starting another ball")
            continue
    
    # Final visualization update
    inner_pcd.points = o3d.utility.Vector3dVector(np.array(simulator.inner_surface_points))
    vis.update_geometry(inner_pcd)
    vis.poll_events()
    vis.update_renderer()
    # Keep window open
    vis.run()
    vis.destroy_window()
    
    return np.array(simulator.inner_surface_points)

if __name__ == "__main__":
    import sys
    # Handle command line arguments
    if len(sys.argv) < 2:
        print("Usage: python ball_scattering.py <path_to_ply_file> [num_balls=100] [ball_radius_factor=0.01]")
        sys.exit(1)

    ply_file = "./ply_files/test point cloud shell_1.ply"
    num_balls = 1 # number of ball instances
    ball_radius_factor = 0.1 # ball radius relative to the size of the entire point cloud (model)

    # Run main simulation
    inner_surface_points = main(ply_file, num_balls, ball_radius_factor)
    # Save results
    output_pcd = o3d.geometry.PointCloud()
    output_pcd.points = o3d.utility.Vector3dVector(inner_surface_points)
    o3d.io.write_point_cloud("inner_surface.ply", output_pcd)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
0


In [ ]:
def find_origin(self):
    """Finds a starting point inside the shell using vector analysis and ray verification"""
    # Get initial centroid and prepare for validation
    centroid = self.point_cloud.get_centroid()
    bbox_diag = self.point_cloud.get_bbox_diagonal()
    k = 100  # Use more neighbors for better normal estimation
    validation_passed = False
    attempts = 0

    while not validation_passed and attempts < 10:
        # Find nearest neighbors to the centroid
        _, indices, _ = self.point_cloud.kdtree.search_knn_vector_3d(centroid, k)
        neighbor_points = self.point_cloud.points[indices]
        
        # Get closest point (likely on surface)
        closest_vector = neighbor_points[0] - centroid
        closest_distance = np.linalg.norm(closest_vector)
        closest_direction = closest_vector / closest_distance
        
        # Calculate surface normal using PCA
        cov = np.cov((neighbor_points - centroid).T)
        _, eigenvectors = np.linalg.eigh(cov)
        normal = eigenvectors[:, 0]  # Smallest eigenvalue direction
        
        # Align normal to point toward centroid (inward)
        if np.dot(normal, closest_direction) < 0:
            normal = -normal
        
        # Move 20% of closest distance inward
        step = 0.2 * closest_distance
        candidate_origin = centroid + normal * step
        centroid = candidate_origin  # Update for next iteration if needed
        
        # Validate position with ray casting
        validation_passed = True
        for ray_direction in [normal, -normal, closest_direction]:
            hit, _ = self.cast_validation_ray(candidate_origin, ray_direction)
            if not hit:
                # Found a ray that didn't hit surface - not inside shell
                validation_passed = False
                break
                
        attempts += 1

    # Fallback to weighted median point if validation fails
    if not validation_passed:
        print("Validation failed, using fallback method")
        k = 500
        _, indices, dists = self.point_cloud.kdtree.search_knn_vector_3d(centroid, k)
        weights = 1.0 / np.sqrt(np.array(dists) + 1e-8)
        candidate_origin = np.average(self.point_cloud.points[indices], axis=0, weights=weights)
        
    self.origin = candidate_origin
    return self.origin

def cast_validation_ray(self, origin, direction, max_distance=None):
    """Casts a validation ray to check if point is inside the shell"""
    if max_distance is None:
        max_distance = self.point_cloud.get_bbox_diagonal() * 2.0
        
    # Define ray: start at origin, extending in direction
    ray_end = origin + direction * max_distance
    
    # Find points along ray path
    query_radius = max_distance * 0.05  # 5% of max distance
    [k, indices, _] = self.point_cloud.kdtree.search_hybrid_vector_3d(
        origin, query_radius, max_nn=100
    )
    
    # Check for intersections
    for idx in indices:
        point = self.point_cloud.points[idx]
        vector_to_point = point - origin
        parallel_dist = np.dot(vector_to_point, direction)
        perp_distance = np.linalg.norm(vector_to_point - parallel_dist * direction)
        
        # Considered a hit if point is in front and within 5% of max_distance
        if parallel_dist > 0 and perp_distance < query_radius and parallel_dist < max_distance:
            return True, point
            
    return False, None

In [2]:
# ====================
# 1. Create a Test Point Cloud
# ====================
def generate_dummy_point_cloud(num_points=1000):
    pcd = o3d.geometry.PointCloud()

    # --- Cube Points (Structured) ---
    cube_points = np.random.uniform(-1, 1, (num_points // 3, 3))
    
    # --- Sphere Points (Semi-Structured) ---
    theta = np.random.uniform(0, 2 * np.pi, num_points // 3)
    phi = np.random.uniform(0, np.pi, num_points // 3)
    r = 1.0
    x = r * np.sin(phi) * np.cos(theta)
    y = r * np.sin(phi) * np.sin(theta)
    z = r * np.cos(phi)
    sphere_points = np.vstack([x, y, z]).T
    
    # --- Random Noise Points ---
    noise_points = np.random.uniform(-1.5, 1.5, (num_points // 3, 3))
    
    # Combine all points
    points = np.vstack([cube_points, sphere_points, noise_points])
    pcd.points = o3d.utility.Vector3dVector(points)
    
    # Assign colors (RGB) based on point type
    colors = np.zeros_like(points)
    colors[:len(cube_points)] = [1, 0, 0]  # Red (Cube)
    colors[len(cube_points):len(cube_points)+len(sphere_points)] = [0, 1, 0]  # Green (Sphere)
    colors[len(cube_points)+len(sphere_points):] = [0, 0, 1]  # Blue (Noise)
    pcd.colors = o3d.utility.Vector3dVector(colors)
    
    return pcd

In [3]:
import open3d as o3d
import numpy as np

# Load your point cloud
pcd = generate_dummy_point_cloud(num_points=100)
points = np.asarray(pcd.points)

# Select a point and direction vector
idx = 0  # Example: first point in the cloud
point = points[idx]
vector = np.array([1.0, 0.5, 0.3])  # Your (x,y,z) vector

# Normalize the vector to determine arrow direction
vector_length = np.linalg.norm(vector)
if vector_length > 0:
    direction = vector / vector_length
else:
    direction = np.array([1, 0, 0])  # Default direction if vector is zero

# Create an arrow (cylinder + cone)
arrow_length = 1  # Scale if needed
arrow = o3d.geometry.TriangleMesh.create_arrow(
    cylinder_radius=0.02,  # Adjust thickness
    cone_radius=0.04,      # Adjust head size
    cylinder_height=arrow_length * 0.7,
    cone_height=arrow_length * 0.3,
)

# Rotate the arrow to align with the vector
if vector_length > 0:
    # Compute rotation between Z-axis and the vector
    z_axis = np.array([0, 0, 1])
    rotation_axis = np.cross(z_axis, direction)
    rotation_angle = np.arccos(np.dot(z_axis, direction))
    rotation_matrix = o3d.geometry.get_rotation_matrix_from_axis_angle(rotation_axis * rotation_angle)
    arrow.rotate(rotation_matrix, center=[0, 0, 0])

# Move the arrow to start at the selected point
arrow.translate(point)

arrow.paint_uniform_color([1, 0, 0])  # RGB values (0-1)

# Visualize
o3d.visualization.draw_geometries(
    [pcd, arrow],
    window_name="Point Cloud with Arrow",
    width=800,
    height=600
)